# B2.0 · What an agentic harness actually is

**Function B — Application Security with an AI SDLC → The Harness that Runs the SDLC**  ·  *AI for Security*

Builds on **[B1.17 · Bonus — Google Mantis, the pipeline in production](https://spbreed.github.io/cyber-commons/lessons/B1.17.html)**.

| | |
|---|---|
| Open-source tooling | LiteLLM, OpenTelemetry |
| Open-weight models | Llama 3.3, GLM-4.6 |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off.

## 1 · The hook

Almost everyone using agents cannot name their harness's verifier, and the answer "the model tells us" means there isn't one. Eight components, named once, and most arguments about agent reliability turn out to be arguments about which of the eight is missing.

## 2 · The framework

```
   the eight components of any harness

   +-----------+   +--------+   +---------+   +-----------+
   |   model   |   |  loop  |   |  tools  |   |  context  |
   +-----------+   +--------+   +---------+   +-----------+
   +-----------+   +--------+   +---------+   +-----------+
   | verifier  |   | budget |   | memory  |   |orchestrator|
   +-----------+   +--------+   +---------+   +-----------+

   the one people cannot name is almost always the verifier
```

**The model is not the system.**

A model is a text generator. Give it tokens, get tokens back. It has no memory
between calls, no ability to act, and no notion of whether it succeeded. Left
alone it cannot read a file, run a scanner or open a pull request.

A **harness** is everything wrapped around that model which turns it into
something that gets work done:

| Component | What it does |
|---|---|
| **The loop** | Decides what happens next — plan, act, observe, decide again — and when to stop |
| **Tools** | The only way the model touches the world |
| **Context management** | What the model sees at each step, assembled from a world much larger than the window |
| **The verifier** | The independent check on whether a step actually succeeded |
| **State & memory** | What survives between steps and between runs |
| **Budgets & stop conditions** | Token, time, cost and action ceilings that bound autonomy |
| **Orchestrator** | Sub-agent spawning, parallelism, delegation depth |
| **Telemetry** | The record that makes a run auditable, replayable and debuggable |

Why this chapter exists: two teams given the **identical model** routinely
differ by an order of magnitude in output quality, purely on harness design.
Most of the capability you attribute to a model is the scaffold around it.

And the security consequence is direct. A harness is itself an autonomous actor
holding credentials and tools — so every harness you build is a system that must
be governed, contained and audited like any other.

## 3 · Build the smallest harness that is still a harness

Eight components, none optional. The model here is a deterministic stand-in — labelled as one — so the scaffold is what you can see.

In [ ]:
from dataclasses import dataclass, field

def stand_in_model(prompt):
    """NOT a language model. A deterministic stub, so the harness is visible.

    It reads the transcript so far to decide what is left to do - which is all
    any agent loop does, minus the part that is hard."""
    if "write_patch" not in prompt:
        return {"tool": "write_patch", "args": {"file": "auth.py"}}
    if "run_tests" not in prompt:
        return {"tool": "run_tests", "args": {}}
    return {"tool": "done", "args": {"claim": "fixed it"}}

WORLD = {"tests_pass": False, "patched": False}

def run_tests(**_):
    # the patch this stub writes does not actually fix the bug
    return {"passed": WORLD["tests_pass"], "failing": [] if WORLD["tests_pass"] else ["test_login"]}
def write_patch(file, **_):
    WORLD["patched"] = True
    return {"wrote": file}
def done(claim, **_):
    return {"claim": claim}

TOOLS = {"run_tests": run_tests, "write_patch": write_patch, "done": done}

@dataclass
class Budget:
    steps: int = 6
    used: int = 0
    def spend(self):
        self.used += 1
        return self.used <= self.steps

def harness(task, verifier=None, budget=None, telemetry=None):
    """loop + tools + context + verifier + state + budget + telemetry."""
    budget = budget or Budget()
    telemetry = telemetry if telemetry is not None else []
    context = [f"TASK: {task}"]                       # context management
    state = {"steps": 0}                              # state
    while budget.spend():                             # budgets / stop conditions
        step = stand_in_model("\n".join(context))     # the model
        tool, args = step["tool"], step["args"]
        result = TOOLS[tool](**args)                  # tools
        state["steps"] += 1
        telemetry.append({"step": state["steps"], "tool": tool, "result": result})
        context.append(f"{tool} -> {result}")
        if tool == "done":
            ok = verifier() if verifier else True     # the verifier
            return {"claimed": True, "verified": ok, "steps": state["steps"],
                    "telemetry": telemetry}
    return {"claimed": False, "verified": False, "steps": state["steps"],
            "telemetry": telemetry}

print("components wired:", ["loop","tools","context","verifier","state",
                            "budget","orchestrator","telemetry"])

## 4 · Run it once with no verifier

In [ ]:
WORLD.update(tests_pass=False, patched=False)
r = harness("fix the failing test_login", verifier=None)
print(f"agent claimed success : {r['claimed']}")
print(f"independently checked : {r['verified']}")
print(f"steps                 : {r['steps']}")
for t in r["telemetry"]:
    print(f"   {t['step']}. {t['tool']:12s}{t['result']}")
print()
print("It reported success. The tests still fail. Nothing in that transcript")
print("is a lie - the agent did write a patch, and then it said it was done.")
assert r["claimed"] and not WORLD["tests_pass"]

## 5 · Where it breaks — the component people leave out

Add the verifier and change nothing else.

In [ ]:
def real_verifier():
    """Ground truth, not self-assessment: run the tests and read the result."""
    return run_tests()["passed"]

WORLD.update(tests_pass=False, patched=False)
r2 = harness("fix the failing test_login", verifier=real_verifier)
print(f"claimed {r2['claimed']}  verified {r2['verified']}")

WORLD.update(tests_pass=True)          # now the fix actually works
r3 = harness("fix the failing test_login", verifier=real_verifier)
print(f"claimed {r3['claimed']}  verified {r3['verified']}")
print()
print("Same model. Same loop. Same tools. The only difference between a harness")
print("that reports the truth and one that reports its own optimism is one")
print("component - and it is the cheapest one in the table.")
assert not r2["verified"] and r3["verified"]

## 6 · The budget is a security control, not a cost control

In [ ]:
def looping_model(prompt):
    return {"tool": "run_tests", "args": {}}          # never finishes

import builtins
_orig = stand_in_model
try:
    globals()["stand_in_model"] = looping_model
    WORLD.update(tests_pass=False)
    r4 = harness("fix it", verifier=real_verifier, budget=Budget(steps=4))
finally:
    globals()["stand_in_model"] = _orig

print(f"ran {r4['steps']} steps, then stopped: claimed={r4['claimed']}")
print()
print("Without the ceiling this runs until something else stops it - a bill, a")
print("rate limit, or an on-call engineer. The budget is what makes 'autonomous'")
print("a bounded word.")
assert r4["steps"] == 4 and not r4["claimed"]

## 7 · Verify — the harness is itself an actor

It holds credentials and calls tools. Score it the way you would score any other non-human identity.

In [ ]:
HARNESS_ACTOR = {
 "identity": "ci-sast-harness",
 "tools": sorted(TOOLS),
 "writes": ["write_patch"],
 "credentials": ["repo:write"],
 "runs_unattended": True,
 "telemetry": True,
}
irreversible = [t for t in HARNESS_ACTOR["writes"]]
print(f"{'property':22s}value")
for k, v in HARNESS_ACTOR.items():
    print(f"{k:22s}{v}")
print()
print(f"tools that change state : {irreversible}")
print(f"unattended              : {HARNESS_ACTOR['runs_unattended']}")
print(f"auditable               : {HARNESS_ACTOR['telemetry']}")
print()
print("Every question you would ask of an agent applies to the thing you just")
print("built to review agents. A harness with repo:write running unattended is")
print("a non-human identity, and it belongs in the inventory in A2 and E1.2.")
assert HARNESS_ACTOR["telemetry"], "an unauditable harness cannot be governed"

## What you just proved

The minimal harness runs and reports success while the tests still fail. Adding one component — a verifier that reads ground truth rather than the agent's own claim — flips `verified` to False on the same run and to True only when the fix genuinely works. A four-step budget stops a looping model. The harness then scores itself as a non-human identity holding repo:write and running unattended.

## Your turn

Take a harness you already run and name its eight components. The one people cannot name is almost always the verifier, and the answer 'the model tells us' means there isn't one.

---

**Next → [B2.1 · Plan–act–verify](https://spbreed.github.io/cyber-commons/lessons/B2.1.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/B2.0.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/B2.0.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*